In [133]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/uciml/iris/Iris.csv
/kaggle/input/datasets/organizations/uciml/iris/database.sqlite


In [134]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder

In [135]:
data = pd.read_csv("/kaggle/input/datasets/organizations/uciml/iris/Iris.csv")
X = data.drop(['Id','Species'], axis=1) 
y = data['Species']
scaler = StandardScaler()

In [136]:
class Softmax:
    def __init__(self,alpha=0.01,iters=1000):
        self.alpha = alpha
        self.iters = iters
        self.weights = None
        self.bias = None
        self.unique_classes = None

    def sm(self,z):
        z = z - np.max(z,axis=1,keepdims=True)
        exp_z = np.exp(z) # Makes array of e^logits for each class
        return ((exp_z)/(np.sum(exp_z,axis=1,keepdims=True)))

    def one_hot(self,y,classes):
        y = np.array(y).flatten().astype(int)
        return np.eye(classes)[y]
    
    def fit(self,X,y):
        X = np.asarray(X)
        m,n = X.shape
        self.unique_classes, y_integers = np.unique(y, return_inverse=True)
        classes = len(self.unique_classes)
        
        self.weights = np.zeros((n,classes))
        self.bias = np.zeros((1,classes))
        y_oh = self.one_hot(y_integers,classes)

        for i in range(self.iters):
            z = X@self.weights + self.bias #logit
            p = self.sm(z) # e^logit

            dJ_dw = (1/m)*(X.T@(p-y_oh))
            dJ_db = (1/m)*(np.sum(p-y_oh,axis=0,keepdims=True))

            self.weights -= self.alpha*dJ_dw
            self.bias -= self.alpha*dJ_db

    def predict(self,X):
        X = np.asarray(X)
        z=X@self.weights + self.bias
        y_hat = self.sm(z)
        indices = np.argmax(y_hat, axis=1)
        return self.unique_classes[indices]
    

In [137]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)
X_train = scaler.fit_transform(X_train)
model = Softmax(alpha=0.01,iters=10000)
model.fit(X_train,y_train)


In [138]:
X_test = scaler.transform(X_test)
predictions = model.predict(X_test)
accuracy = np.mean(predictions==y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")

Test Accuracy: 96.67%
